In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../output/04_alpha_beta_by_Reg.csv')

In [3]:
df['sector'].nunique()

70

# Sector Binning

#### The Colombo Stock Exchange (CSE) utilizes the Global Industry Classification Standard (GICS), organizing listed companies into 20 distinct industry group sector indices.

| # | GICS Sector |
|---|---|
| 1 | Energy |
| 2 | Materials |
| 3 | Capital Goods |
| 4 | Commercial and Professional Services |
| 5 | Transportation |
| 6 | Automobiles and Components |
| 7 | Consumer Durables and Apparel |
| 8 | Consumer Services |
| 9 | Retailing |
| 10 | Food and Staples Retailing |
| 11 | Food, Beverage and Tobacco |
| 12 | Household and Personal Products |
| 13 | Health Care Equipment and Services |
| 14 | Pharmaceuticals, Biotechnology and Life Sciences |
| 15 | Banks |
| 16 | Diversified Financials |
| 17 | Insurance |
| 18 | Telecommunication Services |
| 19 | Utilities |
| 20 | Real Estate |

In [4]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(
    api_key=os.getenv('OPENROUTER_API_KEY'),
    base_url="https://openrouter.ai/api/v1",
)

GICS_SECTORS = [
    "Energy", "Materials", "Capital Goods", "Commercial and Professional Services",
    "Transportation", "Automobiles and Components", "Consumer Durables and Apparel",
    "Consumer Services", "Retailing", "Food and Staples Retailing",
    "Food, Beverage and Tobacco", "Household and Personal Products",
    "Health Care Equipment and Services", "Pharmaceuticals, Biotechnology and Life Sciences",
    "Banks", "Diversified Financials", "Insurance", "Telecommunication Services",
    "Utilities", "Real Estate"
]

def sector_bin(sector: str) -> str:

    prompt = f"""
        There are companies classified into 20 sectors based on the 
        Global Industry Classification Standard (GICS). The sectors are:
        {", ".join(GICS_SECTORS)}

        Which sector does "{sector}" fall into from this list?
        if nothing matches from the list return 'ELSE'
    """.strip()

    response = client.chat.completions.create(
        model="anthropic/claude-opus-4.6",  
        messages=[
            {
                "role": "system",
                "content": """You are a sector classification agent.
                Classify the given sector into one of the 20 GICS sectors.
                Respond with ONLY the exact sector name from the list, nothing else.""".strip()
            },
            {"role": "user", "content": prompt}
        ]
    )

    result = response.choices[0].message.content.strip()


    return result

In [5]:
sectors = np.array(df['sector'].unique().tolist())
map = {sector : sector_bin(sector) for sector in sectors }

In [6]:
df['sector_gics'] = df['sector'].map(map)

In [7]:
df['sector_gics'].unique()

array(['Diversified Financials', 'Insurance',
       'Consumer Durables and Apparel', 'Banks', 'Consumer Services',
       'Capital Goods', 'Food, Beverage and Tobacco', 'Materials',
       'Health Care Equipment and Services', 'Automobiles and Components',
       'Food and Staples Retailing', 'Retailing', 'Real Estate',
       'Commercial and Professional Services', 'ELSE', 'Transportation',
       'Telecommunication Services', 'Utilities', 'Energy',
       'Pharmaceuticals', 'Household and Personal Products'], dtype=object)

Handling the ELSE case

In [8]:
indexes = np.array(df[df['sector_gics']== 'ELSE'].index.values)

In [9]:
else_sector = df.loc[indexes,'sector']
else_sector.value_counts()

sector
Miscellaneous        771
Packaged Software    514
Name: count, dtype: int64

In [10]:
#Getting indexes
idx_mis = np.array(df[df['sector'] == 'Miscellaneous'].index.values)
idx_sof = np.array(df[df['sector'] == 'Packaged Software'].index.values)

#assigning the new categories
df.loc[idx_sof,'sector_gics'] = 'Telecommunication Services'
df.loc[idx_mis,'sector_gics'] = 'other'

#dropping old sector column
df.drop('sector',axis = 1,inplace=True)

#renaming it again for the ease of use
df = df.rename(columns = {'sector_gics' : 'sector'})

In [11]:
df.to_csv('../output/05_sector_binning.csv',index = False)